# 扫雷：从数字提示推出哪些格子安全

## 本章要点

扫雷这游戏，从头到尾都在推理：看着已经翻开的格子上的数字，猜地雷藏在哪儿。这一章，我们把这件活整个交给 Z3。

- 每一格都是一个 `Bool` 变量，它只回答一个问题：这里有没有雷
- 数字格说的“周围数字等于周围地雷数”，原样抄成约束
- 再用**反证法**问两遍：假设有雷会不会矛盾，假设没雷会不会矛盾，靠 `push()` / `check()` / `pop()` 走完一轮

人在盘面上用手指一格一格试，Z3 替我们把每一格都试了一遍。

## 扫雷的规则

盘面上写着的数字，意思很直接：**这一格周围 8 格里埋了几颗雷**。数字格本身是已经翻开的，安全；`.` 则是还没动过的格子。

我们要写的程序只干一件事：拿到一个盘面，把所有**可以断定不是地雷的格子**挑出来。

## 解题思路

一格一格来，每格都试两种假设。

1. 先假装“这一格有地雷”，看它跟已有的约束冲不冲突
   - 冲突（`unsat`）→ 那就不可能真有雷 → **安全格**
2. 再假装“这一格没有地雷”，同样看冲不冲突
   - 冲突（`unsat`）→ 反过来，雷肯定在这儿 → **地雷格**
3. 两种假设都活得下去 → 这一格现在还定不了（运气差的话，翻开就是炸）

这个套路叫反证法：假设放进去，矛盾冒出来，就说明假设本身错了。

麻烦的是假设之间不能互相污染。临时加上的假设，要用 `solver.push()`（先把当前状态存起来）和 `solver.pop()`（用完退回保存点）夹在中间，这样处理下一格时，手上还是干净的约束。

## 代码

代码分成三个函数，各管一件事。

- `get_neighbors()` … 给一个坐标，把周围 8 格列出来
- `solve_board()` … 真正去解盘面，交出安全格和地雷格两份名单
- `show_result()` … 把结果照盘面的样子画回去

In [1]:
from z3 import *

# 8 个方向（上下左右＋斜向）
DIRECTIONS = [
    (-1, -1), (-1, 0), (-1, 1),
    (0, -1),           (0, 1),
    (1, -1),  (1, 0),  (1, 1),
]


def get_neighbors(x, y, h, w):
    """返回格子 (x, y) 周围 8 格中位于盘面内的那些"""
    neighbors = []
    for dx, dy in DIRECTIONS:
        nx, ny = x + dx, y + dy
        if 0 <= nx < w and 0 <= ny < h:
            neighbors.append((nx, ny))
    return neighbors


def solve_board(board_text):
    """求解盘面，返回 (安全格, 地雷格)"""
    rows = board_text.strip().split()
    h, w = len(rows), len(rows[0])

    # 给数字格周围的未翻开格子建一个表示“有地雷”的 Bool 变量
    mine = {}
    for y, row in enumerate(rows):
        for x, v in enumerate(row):
            if v == '.':
                continue                      # 未翻开的格子本身跳过
            for nx, ny in get_neighbors(x, y, h, w):
                if rows[ny][nx] == '.':
                    mine[nx, ny] = Bool(f'mine_{nx}_{ny}')

    # 约束条件：数字格周围为 True 的变量个数 = 这个数字
    solver = Solver()
    for y, row in enumerate(rows):
        for x, v in enumerate(row):
            if v == '.':
                continue
            around = [mine[loc] for loc in get_neighbors(x, y, h, w) if loc in mine]
            if around:
                solver.add(Sum(around) == int(v))

    # 给每个格子加上假设，看是否矛盾
    safe, mines = [], []
    for (x, y), var in mine.items():
        solver.push()
        solver.add(var)                   # 假设：这里有地雷
        if solver.check() == unsat:       # 矛盾 → 不是地雷
            safe.append((x, y))
        solver.pop()

        solver.push()
        solver.add(Not(var))              # 假设：这里没有地雷
        if solver.check() == unsat:       # 矛盾 → 确定是地雷
            mines.append((x, y))
        solver.pop()

    return safe, mines


def show_result(board_text, safe, mines):
    """“#”是安全格，“*”是地雷格"""
    board = [list(row) for row in board_text.strip().split()]
    for x, y in mines:
        board[y][x] = '*'
    for x, y in safe:
        board[y][x] = '#'
    print("\n".join(" ".join(row) for row in board))

## 执行示例

![minesweeper](images/minesweeper01.png)

下面这个盘面，我们不动手，交给代码去推。

In [2]:
board_str = """
...201..................
...201..................
...1013.................
...1102.................
....1011123.............
..21100012...22211122...
..2000012.33212.3111....
..421002.310002.3.122...
....3224.31000112111....
........4.20000001121...
.......43.20000012.11...
.......3122211001.212...
.......202.3.10011213...
......2103.42210001.213.
......1112.43.111233202.
......2.222..311.3..212.
.............2124.4.....
............111.3.......
........................
........................
"""

safe, mines = solve_board(board_str)
print(f"安全格: {len(safe)} 个")
print(f"地雷格: {len(mines)} 个")
show_result(board_str, safe, mines)

安全格: 43 个
地雷格: 54 个
. . * 2 0 1 # . . . . . . . . . . . . . . . . .
. . * 2 0 1 * # . . . . . . . . . . . . . . . .
. . # 1 0 1 3 * . . . . . . . . . . . . . . . .
. . # 1 1 0 2 * # # * # . . . . . . . . . . . .
. # # * 1 0 1 1 1 2 3 # # # * # # # * # # . . .
. . 2 1 1 0 0 0 1 2 * * * 2 2 2 1 1 1 2 2 . . .
. . 2 0 0 0 0 1 2 * 3 3 2 1 2 * 3 1 1 1 * # . .
. * 4 2 1 0 0 2 * 3 1 0 0 0 2 * 3 * 1 2 2 # . .
. . * * 3 2 2 4 * 3 1 0 0 0 1 1 2 1 1 1 * # . .
. . . * * # * * 4 * 2 0 0 0 0 0 0 1 1 2 1 # . .
. . . . . . * 4 3 * 2 0 0 0 0 0 1 2 * 1 1 # . .
. . . . . . * 3 1 2 2 2 1 1 0 0 1 * 2 1 2 * . .
. . . . . * * 2 0 2 * 3 * 1 0 0 1 1 2 1 3 * # .
. . . . . # 2 1 0 3 * 4 2 2 1 0 0 0 1 * 2 1 3 .
. . . . . # 1 1 1 2 * 4 3 * 1 1 1 2 3 3 2 0 2 .
. . . . . # 2 * 2 2 2 * * 3 1 1 * 3 * * 2 1 2 .
. . . . . . . # * # # # * 2 1 2 4 * 4 # . . # .
. . . . . . . . . . . # 1 1 1 * 3 * # # . . . .
. . . . . . . . . . . # # # # # # # . . . . . .
. . . . . . . . . . . . . . . . . . . . . . . .


没翻开的格子一共 111 个。跑完以后，其中 **43 个被判成安全**、**54 个被判成地雷**，都是板上钉钉的结论。剩下的 14 个，两种假设都试过，哪一种都不矛盾，于是只能先搁着。人自己玩的时候也卡在这 14 格上，到最后还是靠猜。

输出里的 `#` 是安全格，`*` 是地雷格。

## 代码讲解

### 1. 创建变量

```python
mine = {}
for y, row in enumerate(rows):
    for x, v in enumerate(row):
        if v == '.':
            continue
        for nx, ny in get_neighbors(x, y, h, w):
            if rows[ny][nx] == '.':
                mine[ny, nx] = Bool(f'mine_{ny}_{nx}')
```

`mine` 是一本字典，`(x, y) → Bool 变量` 就是它的键值关系。变量问的是“这一格有没有地雷”，`True` 有，`False` 没有。

注意，这里只给数字格旁边的未翻开格子建变量。离数字远的格子一个提示都没有，建出来也无从判断。

### 2. 添加约束条件

```python
around = [mine[loc] for loc in get_neighbors(x, y, h, w) if loc in mine]
if around:
    solver.add(Sum(around) == int(v))
```

拿到数字格 `v` 以后，就把“周围那些还没翻开的格子，地雷数正好是 `v`”写成一条约束。

`Sum(around)` 是对 `Bool` 变量求和。Z3 把 `True` 当 1、`False` 当 0 加起来，加出来就是雷的个数，不用自己换算。

`around` 什么时候会是空的？数字格周围已经全部翻开的时候。这种格没有可加的约束，所以用 `if around:` 直接跳过。

### 3. 用反证法判断

```python
solver.push()
solver.add(var)                   # 假设：这里有地雷
if solver.check() == unsat:       # 矛盾 → 不是地雷
    safe.append((x, y))
solver.pop()
```

| 调用 | 作用 |
|---|---|
| `solver.push()` | 把当前的约束存一份，之后才好加临时约束 |
| `solver.add(...)` | 加上这次的假设 |
| `solver.check()` | 带着假设检查有没有解（`unsat` 就是矛盾） |
| `solver.pop()` | 退回存下的状态，把假设扔掉 |

表里最后一行最容易被漏掉。`pop()` 忘了写，上一格的假设就留在求解器里，跟着一起审下一格，答案自然就歪了。`push()` 和 `pop()` 必须成对出现。

## 小结

- 未翻开的格子一旦变成 `Bool` 变量，周围的地雷数就能直接写成 `Sum(...) == 数字`
- 反证法在这儿只有两句话：假设有雷却 `unsat`，那是安全；假设没雷却 `unsat`，那是地雷
- 临时假设得用 `push()` 和 `pop()` 前后夹住，别让它漏到下一次判断里
- 两种假设都不矛盾的格子，就是真的定不了。扫雷里那种一翻就炸的时刻，根子在这里